In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('learn_data_cleansing').getOrCreate()

customers_df = spark.read.format('csv').option("inferSchema","true").option("header","true").load("./customers.csv")

#temp table
customers_df.createOrReplaceTempView("customers_table")

In [ ]:
print(customers_df.printSchema())
customers_df.summary().show()

In [ ]:
from pyspark.sql.functions import isnull, isnan

customers_df.where(isnull('gender') | isnan('gender')).show()
print("Jumlah missing value: ", customers_df.where(isnull('gender') | isnan('gender')).count())

In [ ]:
print("Jumlah duplikasi : ", (customers_df.count() - customers_df.distinct().count()))

In [ ]:
from pyspark.sql.functions import col

new_customers_df = customers_df.withColumn("customer_id", col("customer_id").cast("string"))
new_customers_df = new_customers_df.withColumn("zip_code", col("zip_code").cast("string"))
new_customers_df.printSchema()

In [ ]:
new_customers_df.na.drop()
# new_customers_df.na.drop(how="any") # Menghapus seluruh baris data yang mengandung missing value (walaupun hanya 1)
# customers_df.na.drop(how="all") # Menghapus seluruh baris data jika seluruh baris tersebut bernilai null atau nan
# new_customers_df.na.drop(thresh=2) # Menghapus seluruh baris data jika baris tersebut mengandung minimal 2 missing value

In [ ]:
new_customers_df = new_customers_df.na.drop()
print("Jumlah missing value : ", new_customers_df.where(isnull('gender') | isnan('gender')).count())

In [ ]:
customers_df.na.fill("Prefer not to say", subset=["gender"])

In [ ]:
from pyspark import pandas as ps

customers_df_pandas = ps.read_csv("./customers.csv")
customers_df_pandas["age"].interpolate(method='linear')

In [ ]:
new_customers_df.where(new_customers_df.age > 100).show()

In [ ]:
from pyspark.sql.functions import when

new_customers_df = new_customers_df.withColumn(
    "age", when(new_customers_df.age == 700,70)
    .when(new_customers_df.age == 500,50)
    .otherwise(new_customers_df.age)
)

new_customers_df.summary().show()

In [ ]:
#Drop duplicate
new_customers_df = new_customers_df.dropDuplicates()
print("Jumlah duplikasi: ", (new_customers_df.count() - new_customers_df.distinct().count()))

In [ ]:
#Preprocessing

from pyspark.ml.feature import StandardScaler
from pyspark.ml.feature import VectorAssembler

# Mengubah nilai pada kolom age menjadi vector
assembler = VectorAssembler().setInputCols(['age']).setOutputCol('vec_age')
preprocess_customers_df = assembler.transform(new_customers_df)

standard_scaler = StandardScaler().setInputCol('vec_age').setOutputCol("StandardScaler_age")
preprocess_customers_df = standard_scaler.fit(preprocess_customers_df).transform(preprocess_customers_df)
preprocess_customers_df.show(5)

In [ ]:
#Min Max Scaling
from pyspark.ml.feature import MinMaxScaler

min_max_scaler = MinMaxScaler().setInputCol("vec_age").setOutputCol("MinMaxScaler_age")
preprocess_customers_df = min_max_scaler.fit(preprocess_customers_df).transform(preprocess_customers_df)

preprocess_customers_df.show(5)

In [ ]:
from pyspark.ml.feature import StringIndexer

label_encoder = StringIndexer().setInputCol("gender").setOutputCol("label_gender")
preprocess_customers_df = label_encoder.fit(preprocess_customers_df).transform(preprocess_customers_df)
preprocess_customers_df.show(5)

In [ ]:
  from pyspark.ml.feature import OneHotEncoder

  one_hot_encoder = OneHotEncoder().setInputCol("label_gender").setOutputCol("one_hot_gender")
  one_hot_encoder.fit(preprocess_customers_df).transform(preprocess_customers_df).show(5)

In [ ]:
from pyspark.ml.feature import VectorAssembler, PCA

#Membuat feature buatan untuk digunakan dalam proses PCA
assembler = VectorAssembler().setInputCols(['StandardScaler_age', 'MinMaxScaler_age']).setOutputCol('features')
preprocess_customers_df = assembler.transform(preprocess_customers_df)

pca = PCA().setInputCol("features").setOutputCol("PCA_age").setK(1)
preprocess_customers_df = pca.fit(preprocess_customers_df).transform(preprocess_customers_df)
preprocess_customers_df.show(5, truncate=False)